In [ ]:
# Phase 5 — Monte Carlo Resampling, Convergence & Centroid Portfolios

**Frozen Parameters & Protocol:**
- Empirical Residuals from Ridge Pooled Walk-Forward (74,520 OOS observations)
- Heavy-tailed confirmation: Jarque-Bera p=0.0000 rejected, Student-t MLE nu ≈ 4.3
- Block Bootstrap B = 21 trading days (temporal autocorrelation preservation)
- 37 Quarterly Rebalance Dates x 500 draws per RD (K=500)
- Single-Name Weight Cap w_i <= 10%, Sector Tolerance +/- 3.0 pp
- Selected Centroid: 'mean' aggregation via FR-5 diversification tie-breaker

Panels:
1. Ridge OOS Residual Distribution vs Gaussian & Student-t fits
2. Bootstrap Mode Integrity (Block B=21 contiguity vs iid)
3. Monte Carlo Optimization Loop & Constraint Tracking (37 RDs)
4. Convergence Curve (K=10 -> 500 draws L2 stabilization)
5. Stability A/B Comparison (Centroid vs Equal Weight Baseline)
6. Centroid Aggregation Candidates Comparison (Median vs Mean vs Medoid)
7. Selected Centroid Weight Heatmap Across 37 Rebalance Dates
8. Final Compliance Audit & 2 Hard Asserts (Drift <= 3.0 pp, Cap <= 10.0%, sum(w) = 1.0)


In [ ]:
## Setup & Data Load

In [ ]:
from __future__ import annotations
import pathlib, sys
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

ROOT = pathlib.Path.cwd()
if (ROOT / 'notebooks').exists(): pass
elif (ROOT.parent / 'notebooks').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.monte_carlo import WEIGHT_UPPER_LOCKED, SECTOR_TOLERANCE_LOCKED
from src.optimizer import _sector_group_matrix

PROCESSED = ROOT / 'data' / 'processed'
DOCS = ROOT / 'docs'
sns.set_style('whitegrid')

resid_df = pd.read_csv(PROCESSED / 'phase5_ridge_oos_residuals_empirical.csv')
dist_df = pd.read_csv(PROCESSED / 'phase5_residual_distribution_summary.csv')
boot_df = pd.read_csv(PROCESSED / 'phase5_bootstrap_mode_integrity.csv')
mc_loop_df = pd.read_csv(PROCESSED / 'phase5_mc_loop_summary.csv')
conv_df = pd.read_csv(PROCESSED / 'phase5_convergence_curve.csv')
stab_df = pd.read_csv(PROCESSED / 'phase5_centroid_stability_ab.csv')
comp_panel_df = pd.read_csv(PROCESSED / 'phase5_centroid_comparison_panel.csv')
selected_w_df = pd.read_csv(PROCESSED / 'phase5_selected_centroid_weights.csv')
decision_df = pd.read_csv(PROCESSED / 'phase5_centroid_selection_decision.csv')

universe_df = pd.read_csv(ROOT / 'data' / 'raw' / 'universe_frozen.csv')
tickers = sorted(universe_df['ticker'].tolist())
sector_map_df = pd.read_csv(DOCS / 'nifty50_sector_map.csv')
t2s = dict(zip(sector_map_df['ticker'].astype(str), sector_map_df['sector_provisional'].astype(str)))
ticker_to_sector = {t: t2s[t] for t in tickers if t in t2s}
cnt = Counter(ticker_to_sector.values())
targets = {s: float(c) / sum(cnt.values()) for s, c in cnt.items()}
secs, G_mat = _sector_group_matrix(len(tickers), pd.Series(ticker_to_sector), tickers)
tgt_vec = np.array([float(targets[s]) for s in secs], dtype=float)

print(f'[setup OK] Loaded {len(resid_df)} residuals, {len(selected_w_df)} selected weights.')


In [ ]:
## Panel 1 — Ridge OOS Residual Distribution vs Gaussian & Student-t

In [ ]:
residuals = resid_df['residual'].to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(residuals, bins=100, kde=False, stat='density', ax=ax, alpha=0.4, color='steelblue', label='Empirical Residuals')
x_grid = np.linspace(np.percentile(residuals, 0.5), np.percentile(residuals, 99.5), 500)
mu_norm, sig_norm = np.mean(residuals), np.std(residuals)
ax.plot(x_grid, stats.norm.pdf(x_grid, mu_norm, sig_norm), 'r--', linewidth=1.8, label=f'Normal (sig={sig_norm:.3f})')
df_t = float(dist_df['t_df_mle'].iloc[0])
loc_t = float(dist_df['t_loc_mle'].iloc[0])
scale_t = float(dist_df['t_scale_mle'].iloc[0])
ax.plot(x_grid, stats.t.pdf(x_grid, df=df_t, loc=loc_t, scale=scale_t), 'g-', linewidth=2.0, label=f'Student-t (nu={df_t:.1f})')
ax.set_title(f'Panel 1: Ridge OOS Residuals (kurtosis_excess={dist_df["kurtosis_excess"].iloc[0]:.2f}, JB p={dist_df["jb_p_value"].iloc[0]:.4f})')
ax.legend()
plt.show()


In [ ]:
## Panel 2 — Bootstrap Mode Integrity Table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.axis('off')
table_data = boot_df.values.tolist()
table_cols = boot_df.columns.tolist()
table = ax.table(cellText=table_data, colLabels=table_cols, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.4)
ax.set_title('Panel 2: Bootstrap Mode Integrity Verification (Block B=21 vs iid)')
plt.show()


In [ ]:
## Panel 3 — Monte Carlo Optimization Loop & Constraint Tracking

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
mc_dates = pd.to_datetime(mc_loop_df['rebal_date'])
ax1.plot(mc_dates, mc_loop_df['w_max'] * 100, 'o-', color='purple', label='Max Weight (%)')
ax1.axhline(WEIGHT_UPPER_LOCKED * 100, color='red', linestyle='--', label='10% Cap')
ax1.set_title('Panel 3A: Single-Name Weight Cap by Rebalance Date')
ax1.set_ylabel('Max Weight (%)')
ax1.legend()

ax2.plot(mc_dates, mc_loop_df['sector_drift_max_pp'], 's-', color='teal', label='Max Sector Drift (pp)')
ax2.axhline(SECTOR_TOLERANCE_LOCKED * 100, color='red', linestyle='--', label='3.0pp Tolerance')
ax2.set_title('Panel 3B: Sector Drift by Rebalance Date')
ax2.set_ylabel('Sector Drift (pp)')
ax2.legend()
plt.tight_layout()
plt.show()


In [ ]:
## Panel 4 — Centroid Weight Convergence across Draw Count K

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(conv_df['K_probe'], conv_df['l2_mean_vs_full500_centroid'], 'bo-', linewidth=2.0, label='Mean L2 Distance vs K=500 Centroid')
ax.set_xlabel('Number of MC Draws (K)')
ax.set_ylabel('L2 Distance')
ax.set_title('Panel 4: Centroid Weight Convergence across Draw Count K')
ax.grid(True, linestyle=':', alpha=0.6)
ax.legend()
plt.show()


In [ ]:
## Panel 5 — Stability A/B Comparison (Centroid vs 1/N Equal Weight)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.axis('off')
table_data = stab_df.round(4).astype(str).values.tolist()
table_cols = stab_df.columns.tolist()
table = ax.table(cellText=table_data, colLabels=table_cols, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.4)
ax.set_title('Panel 5: Centroid vs Equal-Weight Stability A/B Comparison')
plt.show()


In [ ]:
## Panel 6 — Centroid Aggregation Candidates Comparison Panel

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(comp_panel_df))
width = 0.25
ax.bar(x - width, comp_panel_df['cap_max_pp'], width, label='Max Weight (pp)', color='steelblue')
ax.bar(x, comp_panel_df['drift_max_pp'], width, label='Max Sector Drift (pp)', color='darkorange')
ax.bar(x + width, comp_panel_df['avg_weight_std_per_rd'] * 100, width, label='Std Weight per RD (x100)', color='seagreen')
ax.axhline(10.0, color='red', linestyle=':', label='10pp Cap')
ax.axhline(3.0, color='gray', linestyle=':', label='3pp Drift Tol')
ax.set_xticks(x)
ax.set_xticklabels(comp_panel_df['aggregation'].tolist())
ax.set_title('Panel 6: Centroid Aggregation Candidates Comparison (Median vs Mean vs Medoid)')
ax.legend()
plt.show()


In [ ]:
## Panel 7 — Selected Centroid Weight Heatmap (37 Dates x 46 Tickers)

In [ ]:
selected_w_df['rebal_date'] = pd.to_datetime(selected_w_df['rebal_date'])
piv = selected_w_df.pivot_table(index='rebal_date', columns='ticker', values='weight').fillna(0.0)
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(piv * 100, ax=ax, cmap='YlGnBu', cbar_kws={'label': 'Weight (%)'}, vmin=0, vmax=10.0)
ax.set_title('Panel 7: Selected Centroid Weights Across 37 Rebalance Dates (0 to 10%)')
plt.tight_layout()
plt.show()


In [ ]:
## Panel 8 — Final Compliance Audit & 2 Hard Asserts

In [ ]:
# Hard Assert 1: Sector drift tolerance (<= 3.000pp)
max_sector_drift_pp = 0.0
for rd, g in selected_w_df.groupby('rebal_date'):
    w = g.set_index('ticker')['weight'].reindex(tickers).fillna(0.0).to_numpy(dtype=float)
    drift = np.abs(G_mat @ w - tgt_vec) * 100.0
    max_sector_drift_pp = max(max_sector_drift_pp, float(np.max(drift)))

print(f'Observed Max Sector Drift: {max_sector_drift_pp:.4f} pp (tolerance <= 3.0000 pp)')
assert max_sector_drift_pp <= (SECTOR_TOLERANCE_LOCKED * 100.0) + 1e-4, f'Sector drift {max_sector_drift_pp:.4f}pp > 3.0pp'
print('>>> HARD ASSERT 1 PASS: Sector Drift <= 3.000 pp across all 37 RDs! <<<')

# Hard Assert 2: Single-name cap (<= 10.0%) and Sum of Weights == 1.0
max_single_weight = float(selected_w_df['weight'].max())
min_single_weight = float(selected_w_df['weight'].min())
sums_per_rd = selected_w_df.groupby('rebal_date')['weight'].sum()
max_sum_err = float(np.max(np.abs(sums_per_rd.to_numpy() - 1.0)))

print(f'Observed Max Single-Name Weight: {max_single_weight*100:.4f}% (cap <= 10.0000%)')
print(f'Observed Min Single-Name Weight: {min_single_weight*100:.6f}% (lower bound >= 0.0%)')
print(f'Observed Max Sum-to-1 Error: {max_sum_err:.2e}')

assert max_single_weight <= WEIGHT_UPPER_LOCKED + 1e-5, f'Single-name weight {max_single_weight*100:.4f}% > 10.0%'
assert min_single_weight >= -1e-5, f'Negative weight found: {min_single_weight}'
assert max_sum_err < 1e-5, f'Weight sum error {max_sum_err} exceeds 1e-5'
print('>>> HARD ASSERT 2 PASS: 10% Single-Name Cap & Sum(w)=1.0 strictly satisfied! <<<')
